In [3]:
import os
from datetime import datetime

def get_min_max_timestamp(folder_path):
    timestamps = []

    for file in os.listdir(folder_path):
        if file.endswith(".jpg"):
            try:
                # remove prefix and extension
                name = file.replace("citra_", "").replace(".jpg", "")
                
                # convert to datetime
                dt = datetime.strptime(name, "%Y-%m-%d_%H%M%S")
                timestamps.append(dt)

            except Exception as e:
                print(f"Skipping file {file}: {e}")

    if not timestamps:
        return None, None

    return min(timestamps), max(timestamps)


folder = r"F:\strwaberry images\back row upated with text\Cropped_Images\down_2\preprocessed_images"

min_time, max_time = get_min_max_timestamp(folder)

print("Earliest Timestamp:", min_time)
print("Latest Timestamp:", max_time)

Earliest Timestamp: 2025-12-12 16:30:01
Latest Timestamp: 2026-01-18 18:45:01


In [4]:
import os
import pandas as pd
from datetime import datetime

def create_timestamp_excel(folder_path, output_excel):
    records = []

    for file in os.listdir(folder_path):
        if file.endswith(".jpg"):
            try:
                name = file.replace(".jpg", "")
                dt = datetime.strptime(name, "%Y-%m-%d_%H-%M")

                records.append({
                    "filename": file,
                    "date": dt.date(),
                    "time": dt.time(),
                    "timestamp": dt
                })

            except Exception as e:
                print(f"Skipping {file}: {e}")

    df = pd.DataFrame(records)

    # sort by time
    df = df.sort_values("timestamp")

    # save excel
    df.to_excel(output_excel, index=False)

    print("Excel saved to:", output_excel)


folder = r"F:\strwaberry images\Front_row_with_text\Cropped_Images_down\down_2\preprocessed_images"
output = r"F:\timestamps.xlsx"

create_timestamp_excel(folder, output)

Excel saved to: F:\timestamps.xlsx


In [2]:
import pandas as pd

def filter_datetime_excel(input_excel, output_excel, start_time, end_time):

    # read excel
    df = pd.read_excel(input_excel)

    # convert Datetime column to datetime format
    df["Date Time"] = pd.to_datetime(df["Date Time"], format="%d-%m-%Y %H:%M")

    # define time range
    start = pd.to_datetime(start_time)
    end = pd.to_datetime(end_time)

    # filter rows
    filtered_df = df[(df["Date Time"] >= start) & (df["Date Time"] <= end)]

    # save new excel
    filtered_df.to_excel(output_excel, index=False)

    print("Filtered Excel saved at:", output_excel)


input_excel = r"FAWN_2025.xlsx"
output_excel = r"down_plants_weather_mapped.xlsx"

start_time = "2025-12-12 16:30:01"
end_time = "2026-01-18 18:45:01"

filter_datetime_excel(input_excel, output_excel, start_time, end_time)

Filtered Excel saved at: down_plants_weather_mapped.xlsx


In [1]:
import pandas as pd

def filter_datetime_excel(input_excel, output_excel, start_time, end_time):

    # read excel
    df = pd.read_excel(input_excel)

    # convert Datetime column to datetime format
    df["Date Time"] = pd.to_datetime(df["Date Time"], format="%d-%m-%Y %H:%M")

    # define time range
    start = pd.to_datetime(start_time)
    end = pd.to_datetime(end_time)

    # filter rows
    filtered_df = df[(df["Date Time"] >= start) & (df["Date Time"] <= end)]

    # save new excel
    filtered_df.to_excel(output_excel, index=False)

    print("Filtered Excel saved at:", output_excel)


input_excel = r"FAWN_2026.xlsx"
output_excel = r"down_plants_weather_mapped_2026.xlsx"

start_time = "2025-12-12 16:30:01"
end_time = "2026-01-18 18:45:01"

filter_datetime_excel(input_excel, output_excel, start_time, end_time)

Filtered Excel saved at: down_plants_weather_mapped_2026.xlsx


In [ ]:
import os
import pandas as pd
from datetime import datetime

def map_images_to_excel(excel_path, image_folder, output_excel):

    # read excel
    df = pd.read_excel(excel_path)
    df["Date Time"] = pd.to_datetime(df["Date Time"], format="%d-%m-%Y %H:%M")

    image_data = []

    # read image timestamps
    for file in os.listdir(image_folder):
        if file.endswith(".jpg"):

            try:
                name = file.replace("citra_", "").replace(".jpg", "")
                img_time = datetime.strptime(name, "%Y-%m-%d_%H%M%S")

                image_data.append((img_time, file))

            except:
                try:
                    name = file.replace(".jpg", "")
                    img_time = datetime.strptime(name, "%Y-%m-%d_%H-%M")

                    image_data.append((img_time, file))
                except:
                    continue

    # convert to dataframe
    img_df = pd.DataFrame(image_data, columns=["img_time", "image_name"])
    img_df = img_df.sort_values("img_time")

    # map nearest timestamp
    df["image_name"] = df["Date Time"].apply(
        lambda x: img_df.iloc[(img_df["img_time"] - x).abs().argsort()[:1]]["image_name"].values[0]
    )

    # save new excel
    df.to_excel(output_excel, index=False)

    print("Mapped Excel saved:", output_excel)


excel_path = r"down_plants_weather_mapped.xlsx"
image_folder = r"back row upated with text\Cropped_Images\down_2\preprocessed_images"
output_excel = r"images_mapped_images_down.xlsx"

map_images_to_excel(excel_path, image_folder, output_excel)

Mapped Excel saved: images_mapped_images_down.xlsx


In [8]:
import os
import pandas as pd
from datetime import datetime

def map_images_with_tolerance(excel_path, image_folder, output_excel):

    # read excel
    df = pd.read_excel(excel_path)
    df["Date Time"] = pd.to_datetime(df["Date Time"], format="%d-%m-%Y %H:%M")
    df = df.sort_values("Date Time")

    image_data = []

    # read image timestamps
    for file in os.listdir(image_folder):

        if file.endswith(".jpg"):
            try:
                name = file.replace("citra_", "").replace(".jpg", "")
                img_time = datetime.strptime(name, "%Y-%m-%d_%H%M%S")
            except:
                try:
                    name = file.replace(".jpg", "")
                    img_time = datetime.strptime(name, "%Y-%m-%d_%H-%M")
                except:
                    continue

            image_data.append([img_time, file])

    img_df = pd.DataFrame(image_data, columns=["img_time", "image_name"])
    img_df = img_df.sort_values("img_time")

    # nearest merge with tolerance ±5 minutes
    mapped = pd.merge_asof(
        df,
        img_df,
        left_on="Date Time",
        right_on="img_time",
        direction="nearest",
        tolerance=pd.Timedelta("5min")
    )

    mapped.drop(columns=["img_time"], inplace=True)

    mapped.to_excel(output_excel, index=False)

    print("Saved:", output_excel)

excel_path = r"down_plants_weather_mapped.xlsx"
image_folder = r"back row upated with text\Cropped_Images\down_2\preprocessed_images"
output_excel = r"images_mapped_images_down.xlsx"

map_images_with_tolerance(excel_path, image_folder, output_excel)

Saved: images_mapped_images_down.xlsx


In [11]:
import pandas as pd

excel_path = r"images_mapped_images_down.xlsx"

df = pd.read_excel(excel_path)

# count mapped images
mapped_count = df["Image_name"].notna().sum()

print("Total mapped images:", mapped_count)

Total mapped images: 1557


In [16]:
df=pd.read_excel("images_mapped_down2.xlsx")

df_filter=df_filtered = df[df["Image_name"].notna()]
# if image name is none  than remove that row.


In [17]:
df_filter

,Station ID,Date Time,Soil Temp (C),Temp @ 60cm (C),Temp @ 2m (C),Temp @ 10m (C),Relative Humidity (%),Dew Point Temp (C),Rainfall Amount (in),Wind Speed (mph),Wind Direction (deg),Solar Radiation (w/m2),Image_name
0,260,2025-12-12 16:45:00,15.60,21.55,21.69,21.16,30.44,3.58,0.00,3.45,229.50,128.50,citra_2025-12-12_164501.jpg
1,260,2025-12-12 17:00:00,15.67,21.67,20.78,21.03,39.15,6.39,0.00,2.14,230.40,89.40,citra_2025-12-12_170001.jpg
2,260,2025-12-12 17:15:00,15.74,19.34,18.13,21.10,48.86,7.21,0.00,0.61,206.10,49.28,citra_2025-12-12_171501.jpg
55,260,2025-12-13 06:30:00,12.38,2.09,3.40,3.28,94.00,2.53,0.00,0.98,87.50,0.01,citra_2025-12-13_063001.jpg
56,260,2025-12-13 06:45:00,12.33,2.02,3.36,3.34,94.00,2.49,0.00,1.22,100.10,0.02,citra_2025-12-13_064501.jpg
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3535,260,2026-01-18 12:45:00,12.78,7.24,7.68,6.89,93.40,6.68,0.01,4.33,55.11,121.40,citra_2026-01-18_124501.jpg
3536,260,2026-01-18 13:00:00,12.72,7.32,7.78,7.00,93.40,6.78,0.01,3.28,17.15,111.40,citra_2026-01-18_130001.jpg
3537,260,2026-01-18 13:15:00,12.66,7.18,7.73,6.94,93.50,6.75,0.00,4.12,36.03,76.42,citra_2026-01-18_131501.jpg
3558,260,2026-01-18 18:30:00,12.30,5.11,5.91,5.37,87.50,4.00,0.00,3.61,313.00,0.08,citra_2026-01-18_183001.jpg


In [ ]:
import pandas as pd
import numpy as np

excel_path = r"F:\only_mapped_images.xlsx"

df = pd.read_excel(excel_path)

# convert datetime
df["Date Time"] = pd.to_datetime(df["Date Time"])

# extract date
df["Date"] = df["Date Time"].dt.date

# temperature column
temp_col = "Temp @ 2m (C)"

# calculate daily Tmin and Tmax
daily_temp = df.groupby("Date")[temp_col].agg(
    Tmin="min",
    Tmax="max"
).reset_index()

# parameters
Tbase = 7
Tmax_limit = 30

# apply rules
daily_temp["Tmax"] = daily_temp["Tmax"].clip(upper=Tmax_limit)
daily_temp["Tmin"] = daily_temp["Tmin"].clip(lower=Tbase)

# calculate GDD
daily_temp["GDD"] = ((daily_temp["Tmax"] + daily_temp["Tmin"]) / 2) - Tbase

# rule: negative GDD = 0
daily_temp["GDD"] = daily_temp["GDD"].clip(lower=0)

# cumulative GDD
daily_temp["Cumulative_GDD"] = daily_temp["GDD"].cumsum()

print(daily_temp.head())

# save
daily_temp.to_excel(r"F:\strawberry_gdd.xlsx", index=False)

In [18]:
file = r"images_mapped_down2.xlsx"

df = pd.read_excel(file)

# convert datetime
df["Date Time"] = pd.to_datetime(df["Date Time"])

# extract date
df["date"] = df["Date Time"].dt.date

# use 2m temperature
temp_col = "Temp @ 2m (C)"

# calculate daily max and min
daily_temp = df.groupby("date")[temp_col].agg(["max","min"]).reset_index()
daily_temp.columns = ["date","Tmax","Tmin"]

Tbase = 7
Tmax_cap = 30

# apply rules
daily_temp["Tmax"] = daily_temp["Tmax"].clip(upper=Tmax_cap)
daily_temp["Tmin"] = daily_temp["Tmin"].clip(lower=Tbase)

# GDD formula
daily_temp["GDD"] = ((daily_temp["Tmax"] + daily_temp["Tmin"]) / 2) - Tbase

# no negative GDD
daily_temp["GDD"] = daily_temp["GDD"].clip(lower=0)

print(daily_temp)

          date   Tmax   Tmin     GDD
0   2025-12-12  21.69   7.00   7.345
1   2025-12-13  25.09   7.00   9.045
2   2025-12-14  24.89   7.26   9.075
3   2025-12-15  13.32   7.00   3.160
4   2025-12-16  20.02   7.00   6.510
5   2025-12-17  24.14   7.86   9.000
6   2025-12-18  25.09  10.79  10.940
7   2025-12-19  23.81   7.00   8.405
8   2025-12-20  22.49   7.00   7.745
9   2025-12-21  24.28   7.00   8.640
10  2025-12-22  22.97   7.00   7.985
11  2025-12-23  26.26   9.19  10.725
12  2025-12-24  27.26   8.68  10.970
13  2025-12-25  26.77  10.93  11.850
14  2025-12-26  26.22  10.61  11.415
15  2025-12-27  25.46  15.98  13.720
16  2025-12-28  25.16  13.69  12.425
17  2025-12-29  24.86  10.16  10.510
18  2025-12-30  12.47   7.00   2.735
19  2025-12-31  13.63   7.00   3.315
20  2026-01-01  19.52   7.00   6.260
21  2026-01-02  21.51   7.00   7.255
22  2026-01-03  25.62  13.82  12.720
23  2026-01-04  15.36  11.90   6.630
24  2026-01-05  21.51  10.91   9.210
25  2026-01-06  25.46   8.65  10.055
2

In [19]:
file = r"images_mapped_down2.xlsx"
df = pd.read_excel(file)

# convert datetime
df["Date Time"] = pd.to_datetime(df["Date Time"])

# extract date
df["date"] = df["Date Time"].dt.date

temp_col = "Temp @ 2m (C)"

# calculate daily Tmax and Tmin
daily = df.groupby("date")[temp_col].agg(["max","min"]).reset_index()
daily.columns = ["date","Tmax","Tmin"]

# GDD parameters
Tbase = 7
Tmax_cap = 30

# apply rules
daily["Tmax"] = daily["Tmax"].clip(upper=Tmax_cap)
daily["Tmin"] = daily["Tmin"].clip(lower=Tbase)

daily["GDD"] = ((daily["Tmax"] + daily["Tmin"]) / 2) - Tbase
daily["GDD"] = daily["GDD"].clip(lower=0)

# merge back into main dataset
df_final = df.merge(daily, on="date", how="left")
df_final

,Station ID,Date Time,Soil Temp (C),Temp @ 60cm (C),Temp @ 2m (C),Temp @ 10m (C),Relative Humidity (%),Dew Point Temp (C),Rainfall Amount (in),Wind Speed (mph),Wind Direction (deg),Solar Radiation (w/m2),Image_name,date,Tmax,Tmin,GDD
0,260,2025-12-12 16:45:00,15.60,21.55,21.69,21.16,30.44,3.58,0.0,3.45,229.5,128.50,citra_2025-12-12_164501.jpg,2025-12-12,21.69,7.0,7.345
1,260,2025-12-12 17:00:00,15.67,21.67,20.78,21.03,39.15,6.39,0.0,2.14,230.4,89.40,citra_2025-12-12_170001.jpg,2025-12-12,21.69,7.0,7.345
2,260,2025-12-12 17:15:00,15.74,19.34,18.13,21.10,48.86,7.21,0.0,0.61,206.1,49.28,citra_2025-12-12_171501.jpg,2025-12-12,21.69,7.0,7.345
3,260,2025-12-12 17:30:00,15.78,15.69,15.99,18.07,59.45,8.10,0.0,0.99,105.6,10.10,NaN,2025-12-12,21.69,7.0,7.345
4,260,2025-12-12 17:45:00,15.81,12.87,14.28,15.60,64.40,7.66,0.0,1.11,160.9,3.27,NaN,2025-12-12,21.69,7.0,7.345
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3555,260,2026-01-18 17:45:00,12.39,6.08,6.67,6.16,87.40,4.73,0.0,6.97,317.4,7.33,NaN,2026-01-18,14.09,7.0,3.545
3556,260,2026-01-18 18:00:00,12.37,5.71,6.29,5.87,86.20,4.15,0.0,6.90,311.5,4.26,NaN,2026-01-18,14.09,7.0,3.545
3557,260,2026-01-18 18:15:00,12.33,5.29,6.03,5.57,86.10,3.88,0.0,4.86,310.5,1.22,NaN,2026-01-18,14.09,7.0,3.545
3558,260,2026-01-18 18:30:00,12.30,5.11,5.91,5.37,87.50,4.00,0.0,3.61,313.0,0.08,citra_2026-01-18_183001.jpg,2026-01-18,14.09,7.0,3.545


In [ ]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.express as px
from datetime import datetime

folder = r"F:\strwaberry images\back row upated with text\Cropped_Images\down_2\preprocessed_images"

records = []

for path in sorted(Path(folder).glob("*.jpg")):

    # ---------- READ IMAGE ----------
    img = cv2.imread(str(path))
    if img is None:
        continue

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # ---------- GREEN MASK ----------
    lower_green = np.array([35, 40, 40])
    upper_green = np.array([90, 255, 255])

    mask = cv2.inRange(hsv, lower_green, upper_green)

    kernel = np.ones((5,5),np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    green_pixels = cv2.countNonZero(mask)

    # ---------- EXTRACT FULL DATETIME ----------
    # citra_2025-12-19_144501
    name = path.stem.replace("citra_", "")
    dt = datetime.strptime(name, "%Y-%m-%d_%H%M%S")

    records.append([dt, green_pixels])



In [20]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

# load your main dataset
# df_final = pd.read_excel(r"F:\dataset_with_gdd.xlsx")

folder = r"F:\strwaberry images\back row upated with text\Cropped_Images\down_2\preprocessed_images"

green_dict = {}

for path in Path(folder).glob("*.jpg"):

    img = cv2.imread(str(path))
    if img is None:
        continue

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    lower_green = np.array([35,40,40])
    upper_green = np.array([90,255,255])

    mask = cv2.inRange(hsv, lower_green, upper_green)

    kernel = np.ones((5,5),np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    green_pixels = cv2.countNonZero(mask)

    green_dict[path.name] = green_pixels


# add green_pixels column using image_name
df_final["green_pixels"] = df_final["Image_name"].map(green_dict)
df_final

,Station ID,Date Time,Soil Temp (C),Temp @ 60cm (C),Temp @ 2m (C),Temp @ 10m (C),Relative Humidity (%),Dew Point Temp (C),Rainfall Amount (in),Wind Speed (mph),Wind Direction (deg),Solar Radiation (w/m2),Image_name,date,Tmax,Tmin,GDD,green_pixels
0,260,2025-12-12 16:45:00,15.60,21.55,21.69,21.16,30.44,3.58,0.0,3.45,229.5,128.50,citra_2025-12-12_164501.jpg,2025-12-12,21.69,7.0,7.345,129763.0
1,260,2025-12-12 17:00:00,15.67,21.67,20.78,21.03,39.15,6.39,0.0,2.14,230.4,89.40,citra_2025-12-12_170001.jpg,2025-12-12,21.69,7.0,7.345,133491.0
2,260,2025-12-12 17:15:00,15.74,19.34,18.13,21.10,48.86,7.21,0.0,0.61,206.1,49.28,citra_2025-12-12_171501.jpg,2025-12-12,21.69,7.0,7.345,131782.0
3,260,2025-12-12 17:30:00,15.78,15.69,15.99,18.07,59.45,8.10,0.0,0.99,105.6,10.10,NaN,2025-12-12,21.69,7.0,7.345,NaN
4,260,2025-12-12 17:45:00,15.81,12.87,14.28,15.60,64.40,7.66,0.0,1.11,160.9,3.27,NaN,2025-12-12,21.69,7.0,7.345,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3555,260,2026-01-18 17:45:00,12.39,6.08,6.67,6.16,87.40,4.73,0.0,6.97,317.4,7.33,NaN,2026-01-18,14.09,7.0,3.545,NaN
3556,260,2026-01-18 18:00:00,12.37,5.71,6.29,5.87,86.20,4.15,0.0,6.90,311.5,4.26,NaN,2026-01-18,14.09,7.0,3.545,NaN
3557,260,2026-01-18 18:15:00,12.33,5.29,6.03,5.57,86.10,3.88,0.0,4.86,310.5,1.22,NaN,2026-01-18,14.09,7.0,3.545,NaN
3558,260,2026-01-18 18:30:00,12.30,5.11,5.91,5.37,87.50,4.00,0.0,3.61,313.0,0.08,citra_2026-01-18_183001.jpg,2026-01-18,14.09,7.0,3.545,153805.0


In [23]:
# remove rows without image / green pixels
df = df.dropna(subset=["green_pixels"])

# scatter plot
fig = px.scatter(
    df,
    x="GDD",
    y="green_pixels",
    trendline="ols",
    title="Daily GDD vs Strawberry Green Pixel Area"
)

fig.show()

In [24]:
df["Date Time"] = pd.to_datetime(df["Date Time"])
df["date"] = df["Date Time"].dt.date

df = df.dropna(subset=["green_pixels"])

# aggregate by day
daily = df.groupby("date").agg({
    "green_pixels":"mean",
    "GDD":"first"
}).reset_index()

fig = px.scatter(
    daily,
    x="GDD",
    y="green_pixels",
    title="Daily GDD vs Average Green Pixel Area",
    labels={
        "GDD":"Daily Growing Degree Days",
        "green_pixels":"Average Green Pixel Area"
    }
)

fig.show()

In [25]:
df["Date Time"] = pd.to_datetime(df["Date Time"])
df["date"] = df["Date Time"].dt.date

df = df.dropna(subset=["green_pixels"])

# aggregate by day
daily = df.groupby("date").agg({
    "green_pixels":"mean",
    "GDD":"first"
}).reset_index()

fig = px.scatter(
    daily,
    x="GDD",
    y="green_pixels",
    title="Daily GDD vs Average Green Pixel Area",
    labels={
        "GDD":"Daily Growing Degree Days",
        "green_pixels":"Average Green Pixel Area"
    }
)
fig.show()

In [26]:
daily = df.groupby("date").agg({
    "green_pixels":"mean",
    "GDD":"first"
}).reset_index()

# plot
fig = px.scatter(
    daily,
    x="date",
    y="green_pixels",
    color="GDD",
    title="Strawberry Growth Over Time (Green Pixels) with GDD",
    labels={
        "date":"Date",
        "green_pixels":"Average Green Pixel Area",
        "GDD":"Daily GDD"
    }
)

fig.update_traces(marker=dict(size=9))

fig.show()

In [ ]:
df_final.to_excel(r"F:\final_dataset_down2_with_gdd_and_green_pixels.xlsx", index=False)

In [3]:
import pandas as pd
df=pd.read_excel('final_dataset_back_down2_with_gdd_and_green_pixels.xlsx')
daily = df.groupby("date").agg({
    "Solar Radiation (w/m2)": "sum",
    "green_pixels": "sum"
}).reset_index()
import plotly.express as px

fig = px.scatter(
    daily,
    x="Solar Radiation (w/m2)",
    y="green_pixels",
    trendline="ols",
    title="Daily Solar Radiation vs Plant Size"
)

fig.show()

In [4]:
import pandas as pd
df=pd.read_excel('final_dataset_back_down2_with_gdd_and_green_pixels.xlsx')
daily = df.groupby("date").agg({
    "Solar Radiation (w/m2)": "mean",
    "green_pixels": "mean"
}).reset_index()
import plotly.express as px

fig = px.scatter(
    daily,
    x="Solar Radiation (w/m2)",
    y="green_pixels",
    trendline="ols",
    title="Daily Solar Radiation vs Plant Size"
)

fig.show()

In [6]:
import pandas as pd
import plotly.graph_objects as go

df = pd.read_excel('final_dataset_back_down2_with_gdd_and_green_pixels.xlsx')

# ensure date column is datetime
df["date"] = pd.to_datetime(df["date"])

daily = df.groupby("date").agg({
    "Solar Radiation (w/m2)": "mean",
    "green_pixels": "mean"
}).reset_index()

fig = go.Figure()

# Plant size line
fig.add_trace(go.Scatter(
    x=daily["date"],
    y=daily["green_pixels"],
    mode='lines+markers',
    name='Plant Size (Green Pixels)'
))

# Radiation line (secondary axis)
fig.add_trace(go.Scatter(
    x=daily["date"],
    y=daily["Solar Radiation (w/m2)"],
    mode='lines+markers',
    name='Daily Radiation',
    yaxis='y2'
))

fig.update_layout(
    title="Daily Radiation and Plant Size Over Time of back setup down 2 plant",
    xaxis_title="Date",
    yaxis=dict(title="Green Pixels"),
    yaxis2=dict(
        title="Daily Radiation (W/m² mean)",
        overlaying='y',
        side='right'
    )
)

fig.show()

In [1]:
!pip install ultralytics 

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 19.8 MB/s  0:00:00
   ---------------------------------------- 0.0/810.4 kB ? eta -:--:--
   ---------------------------------------- 810.4/810.4 kB 34.3 MB/s  0:00:00
   ---------------------------------------- 0.0/45.7 MB ? eta -:--:--
   ----- ---------------------------------- 6.8/45.7 MB 34.9 MB/s eta 0:00:02
   -------------- ------------------------- 16.0/45.7 MB 40.2 MB/s eta 0:00:01
   --------------------- ------------------ 24.9/45.7 MB 40.4 MB/s eta 0:00:01
   ----------------------------- ---------- 33.6/45.7 MB 40.9 MB/s eta 0:00:01
   ------------------------------------- -- 42.5/45.7 MB 41.5 MB/s eta 0:00:01
   ---------------------------------------- 45.7/45.7 MB 38.2 MB/s  0:00:01
   ---------------------------------------- 0.0/113.8 MB ? eta -:--:--
   --- ------------------------------------ 8.7/113.8 MB 44.6 MB/s eta 0:00:03
   ------ --------

In [5]:
from ultralytics import YOLO
import pandas as pd
import os

# Load model
model = YOLO("best.pt")

image_folder = r"F:\strwaberry images\back row upated with text\Cropped_Images\down_2\preprocessed_images"
results_list = []
failed_images = []

for img in os.listdir(image_folder):

    img_path = os.path.join(image_folder, img)

    try:
        # Run inference
        results = model(img_path)

        class_counts = {}
        total = 0

        for r in results:
            if r.boxes is None:
                continue

            boxes = r.boxes.cls.tolist()

            for cls in boxes:
                class_name = model.names[int(cls)]

                class_counts[class_name] = class_counts.get(class_name, 0) + 1
                total += 1

        row = {
            "image_name": img,
            "total_objects": total
        }

        row.update(class_counts)
        results_list.append(row)

    except Exception as e:
        print(f"Error processing {img}: {e}")
        failed_images.append(img)
        continue

# Convert to dataframe
df = pd.DataFrame(results_list).fillna(0)

# Save detection results
df.to_excel("yolo_results_back.xlsx", index=False)

# Save failed images list
if failed_images:
    failed_df = pd.DataFrame({"failed_images": failed_images})
    failed_df.to_excel("failed_images_back.xlsx", index=False)

print("Processing complete.")
print(f"Successful images: {len(results_list)}")
print(f"Failed images: {len(failed_images)}")


image 1/1 F:\strwaberry images\back row upated with text\Cropped_Images\down_2\preprocessed_images\citra_2025-12-12_163001.jpg: 448x640 (no detections), 474.2ms
Speed: 4.0ms preprocess, 474.2ms inference, 1.0ms postprocess per image at shape (1, 3, 448, 640)

image 1/1 F:\strwaberry images\back row upated with text\Cropped_Images\down_2\preprocessed_images\citra_2025-12-12_164501.jpg: 448x640 1 G, 447.1ms
Speed: 3.0ms preprocess, 447.1ms inference, 1.5ms postprocess per image at shape (1, 3, 448, 640)

image 1/1 F:\strwaberry images\back row upated with text\Cropped_Images\down_2\preprocessed_images\citra_2025-12-12_170001.jpg: 448x640 1 G, 433.3ms
Speed: 3.2ms preprocess, 433.3ms inference, 2.1ms postprocess per image at shape (1, 3, 448, 640)

image 1/1 F:\strwaberry images\back row upated with text\Cropped_Images\down_2\preprocessed_images\citra_2025-12-12_171501.jpg: 448x640 1 P, 431.8ms
Speed: 2.8ms preprocess, 431.8ms inference, 1.6ms postprocess per image at shape (1, 3, 448, 6

In [2]:
from ultralytics import YOLO
import pandas as pd
import os

# Load YOLO model
model = YOLO("best.pt")   # your trained model

image_folder = r"F:\strwaberry images\back row upated with text\Cropped_Images\down_2\preprocessed_images"  # folder containing images

results_list = []

for img in os.listdir(image_folder):
    
    img_path = os.path.join(image_folder, img)

    results = model(img_path)

    class_counts = {}
    total = 0

    for r in results:
        boxes = r.boxes.cls.tolist()

        for cls in boxes:
            class_name = model.names[int(cls)]

            if class_name not in class_counts:
                class_counts[class_name] = 0

            class_counts[class_name] += 1
            total += 1

    row = {"image_name": img, "total_objects": total}

    for k, v in class_counts.items():
        row[k] = v

    results_list.append(row)

# df = pd.DataFrame(results_list)

# # Fill missing class counts with 0
# df = df.fillna(0)

# df.to_excel("yolo_results.xlsx", index=False)

# print("Results saved to yolo_results.xlsx")


image 1/1 F:\strwaberry images\back row upated with text\Cropped_Images\down_2\preprocessed_images\citra_2025-12-12_163001.jpg: 448x640 (no detections), 308.7ms
Speed: 6.1ms preprocess, 308.7ms inference, 1.1ms postprocess per image at shape (1, 3, 448, 640)

image 1/1 F:\strwaberry images\back row upated with text\Cropped_Images\down_2\preprocessed_images\citra_2025-12-12_164501.jpg: 448x640 1 G, 283.4ms
Speed: 2.9ms preprocess, 283.4ms inference, 1.4ms postprocess per image at shape (1, 3, 448, 640)

image 1/1 F:\strwaberry images\back row upated with text\Cropped_Images\down_2\preprocessed_images\citra_2025-12-12_170001.jpg: 448x640 1 G, 272.3ms
Speed: 1.7ms preprocess, 272.3ms inference, 1.0ms postprocess per image at shape (1, 3, 448, 640)

image 1/1 F:\strwaberry images\back row upated with text\Cropped_Images\down_2\preprocessed_images\citra_2025-12-12_171501.jpg: 448x640 1 P, 263.9ms
Speed: 1.9ms preprocess, 263.9ms inference, 0.8ms postprocess per image at shape (1, 3, 448, 6

FileNotFoundError: file 'citra_2025-12-12_163001.jpg' does not exist

In [3]:
results_list

[{'image_name': 'citra_2025-12-12_163001.jpg', 'total_objects': 0},
 {'image_name': 'citra_2025-12-12_164501.jpg', 'total_objects': 1, 'G': 1},
 {'image_name': 'citra_2025-12-12_170001.jpg', 'total_objects': 1, 'G': 1},
 {'image_name': 'citra_2025-12-12_171501.jpg', 'total_objects': 1, 'P': 1},
 {'image_name': 'citra_2025-12-13_063001.jpg', 'total_objects': 1, 'R': 1},
 {'image_name': 'citra_2025-12-13_064501.jpg', 'total_objects': 0},
 {'image_name': 'citra_2025-12-13_070001.jpg', 'total_objects': 1, 'G': 1},
 {'image_name': 'citra_2025-12-13_071501.jpg', 'total_objects': 0},
 {'image_name': 'citra_2025-12-13_073001.jpg', 'total_objects': 0},
 {'image_name': 'citra_2025-12-13_074501.jpg',
  'total_objects': 3,
  'FL': 1,
  'W': 2},
 {'image_name': 'citra_2025-12-13_080001.jpg',
  'total_objects': 2,
  'W': 1,
  'G': 1},
 {'image_name': 'citra_2025-12-13_081501.jpg', 'total_objects': 0},
 {'image_name': 'citra_2025-12-13_083001.jpg', 'total_objects': 1, 'FL': 1},
 {'image_name': 'citra

In [ ]:
df = pd.DataFrame(results_list)

# Fill missing class counts with 0
df = df.fillna(0)
df
df.to_excel("yolo_results_back.xlsx", index=False)

print("Results saved to yolo_results.xlsx")